# TrustShift-IDS — Revision Round 2 (post hoc, evaluation-study framing)
Run R0 → R1b → R6 → R7 → R8 in order (R0 is identical to Round 1). Nothing frozen is modified; outputs go to `V2_adaptive_reliability/revision_R1/`.

### CELL R0 — mount, provenance gate, shared helpers

In [ ]:
# CELL R0 — mount, provenance gate, shared helpers
from google.colab import drive
drive.mount("/content/drive")

import os, sys, json, hashlib, platform, gc, time, shutil
import numpy as np, pandas as pd, joblib
import sklearn, scipy, xgboost

BASE = "/content/drive/MyDrive/TrustShift_IDS"
V2   = f"{BASE}/V2_adaptive_reliability"
OUT  = f"{V2}/revision_R1"
os.makedirs(OUT, exist_ok=True)

P = {
  "model_b":      f"{BASE}/models/xgboost/XGB_binary_seed42.joblib",
  "model_m":      f"{BASE}/models/xgboost/XGB_multiclass_seed42.joblib",
  "policy_json":  f"{V2}/adaptive_policy/V2_STEP_10B_FROZEN_PRIMARY_RELIABILITY_POLICY.json",
  "cal":          f"{BASE}/data/processed/CICIDS2017_68F_imputed/CICIDS2017_calibration_68F_imputed.parquet",
  "iid":          f"{BASE}/data/processed/CICIDS2017_68F_imputed/CICIDS2017_iid_test_68F_imputed.parquet",
  "s3_protocol":  f"{V2}/controlled_shifts/S3_corruption/V2_STEP_6D_S3_CORRUPTION_REALIZATION.json",
  "medians":      f"{V2}/controlled_shifts/Level2_compounds/reconstruction_reference/V2_FULL_TRAINING_FEATURE_MEDIANS_68F.npy",
  "s4_recon":     f"{V2}/artifact_audits/V2_STEP_7B_S4_IMPORTANT_EXACT_RECONSTRUCTION.csv",
  "s3_monitor":   f"{V2}/controlled_shifts/S3_corruption/V2_S3_PAIRED_PRIMARY_SSS_ALL_CONDITIONS.csv",
  "s4_monitor":   f"{V2}/controlled_shifts/S4_missing_features/V2_S4_IMPORTANT_PRIMARY_SSS_windows.csv",
  "s3_frozen_out":f"{V2}/adaptive_policy/primary_evaluation/S3/V2_STEP_11B_S3_PRIMARY_ADAPTIVE_WINDOW_RESULTS.csv",
  "s4_frozen_out":f"{V2}/adaptive_policy/primary_evaluation/S4/V2_STEP_11E_S4_PRIMARY_ADAPTIVE_WINDOW_RESULTS.csv",
  "ms_monitor":   f"{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13C_MULTI_SEED_LABEL_FREE_MONITORING_WINDOWS.csv",
  "ms_frozen_out":f"{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13D_MULTI_SEED_FROZEN_POLICY_WINDOW_OUTCOMES.csv",
  "s1_monitor":   f"{V2}/deployment_evaluation/V2_S1_EXTERNAL_PRIMARY_SSS_windows.csv",
  "s1_frozen_out":f"{V2}/deployment_evaluation/V2_S1_EXTERNAL_SSS_RELIABILITY_ASSOCIATION.csv",
  "s1_pred_dir":  f"{BASE}/results/shift_evaluation/S1_external/predictions",
  "null_sss":     f"{V2}/shift_monitoring/V2_source_null_PRIMARY_SSS.csv",
  "norm_params":  f"{V2}/shift_monitoring/V2_monitoring_normalization_parameters.csv",
  "ablation_win": f"{V2}/ablation_evaluation/primary_monitor_family_ablation/V2_STEP_14B_LABEL_FREE_FAMILY_ABLATION_WINDOWS.csv",
  "ablation_out": f"{V2}/ablation_evaluation/primary_monitor_family_ablation/V2_STEP_14C_FAMILY_ABLATION_WINDOW_OUTCOMES.csv",
  "l2_out":       f"{V2}/adaptive_policy/primary_evaluation/Level2_compounds/V2_STEP_11F_C_LEVEL2_FROZEN_ADAPTIVE_WINDOW_RESULTS.csv",
  "loao_fixed":   f"{BASE}/results/shift_evaluation/S2_LOAO/Step_12_3E_LOAO_unseen_fixed_coverage.csv",
}
for s in [1, 7, 21, 42, 84]:
    P[f"ms_X_{s}"] = f"{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13B_SEED_{s}_FLOAT64_STREAM_FEATURES.npz"

def sha256(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(block), b""):
            h.update(b)
    return h.hexdigest()

missing = [k for k, p in P.items() if not os.path.exists(p)]
print("Missing artifacts:", missing if missing else "none")
if missing:
    raise FileNotFoundError("Paste this output back; do not continue.")

manifest = {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "python": sys.version.split()[0], "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
            "scipy": scipy.__version__, "xgboost": xgboost.__version__, "artifacts": {}}
for k, p in P.items():
    if os.path.isdir(p):
        continue
    sz = os.path.getsize(p)
    manifest["artifacts"][k] = {"path": p, "bytes": sz, "sha256": sha256(p) if sz < 3e9 else None}

if xgboost.__version__ != "3.4.1":
    print(f"WARNING: frozen runs used XGBoost 3.4.1; this runtime has {xgboost.__version__}. "
          "The regression gates below will tell us whether predictions still match.")

# ---- frozen constants (manuscript Section 3.4 / 3.8) -------------------------------------
T_B, T_M = 0.9744155346869672, 1.1709901909513676
EPS = 0.01
WINDOW = 5000
THR = {"normal":   {"binary": 0.49778103828430176, "multiclass": 0.49221134185791016},
       "elevated": {"binary": 0.0012131255763865,  "multiclass": 0.0016204167165527},
       "severe":   {"binary": 4.613128198549532e-05, "multiclass": 8.730944055290735e-05}}
pol = json.load(open(P["policy_json"]))
for st, v in pol["thresholds"].items():
    for task in ["binary", "multiclass"]:
        assert abs(float(v[task]) - THR[st.lower()][task]) < 1e-15, (st, task)
print("[R0] Frozen V2.10B thresholds match manuscript values: PASS")

EXPECTED_MEDIAN_FILE_SHA = "44fa27a24dcfbb9274bbe9f5cdc72aa49069d3566ac34b9dd03dee790ed5d4e7"
assert manifest["artifacts"]["medians"]["sha256"] == EXPECTED_MEDIAN_FILE_SHA, "median SHA mismatch"
training_median = np.load(P["medians"], allow_pickle=False)
print("[R0] Frozen training-median vector SHA: PASS")

FEATURE_ORDER = list(json.load(open(P["s3_protocol"]))["feature_order"])
_constants = {"F31", "F33", "F55", "F56", "F57", "F58", "F59", "F60"}
_expected = [f"F{i:02d}" for i in range(1, 77) if f"F{i:02d}" not in _constants]
assert FEATURE_ORDER == _expected and len(FEATURE_ORDER) == 68
print("[R0] 68-feature order: PASS")

FAMILY_MAP = {"Benign": 0, "DDoS": 1, "DoS": 2, "Brute Force": 3, "Web Attack": 4,
              "Botnet": 5, "Infiltration": 6, "Port Scan": 7, "Heartbleed": 8}

model_b = joblib.load(P["model_b"]); model_m = joblib.load(P["model_m"])

def temperature_scale(probs, T):
    p = np.clip(np.asarray(probs, dtype=np.float64), 1e-15, 1.0)
    z = np.log(p) / T
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def predict(X):
    pb = temperature_scale(model_b.predict_proba(X), T_B)
    pm = temperature_scale(model_m.predict_proba(X), T_M)
    return {"binary": (1.0 - pb.max(1), pb.argmax(1).astype(np.int64)),
            "multiclass": (1.0 - pm.max(1), pm.argmax(1).astype(np.int64))}

def labels_from_frame(df):
    y_m = df["attack_family"].map(FAMILY_MAP)
    assert not y_m.isna().any(), "unmapped attack family"
    y_m = y_m.to_numpy(np.int64)
    raw = df["binary_label"]
    if np.issubdtype(raw.dtype, np.number):
        y_b = raw.to_numpy(np.int64)
    else:
        y_b = raw.astype(str).str.strip().str.lower().map({"benign": 0, "attack": 1, "0": 0, "1": 1}).to_numpy(np.int64)
    assert np.array_equal(y_b, (y_m != 0).astype(np.int64))
    return {"binary": y_b, "multiclass": y_m}

def window_record(risk, pred, y, thr):
    """Counts for one window/threshold. 'Attack' = true non-benign; missed attack = attack predicted benign (class 0)."""
    acc = risk <= thr
    err = pred != y
    att = y != 0
    return {"n": int(len(y)), "acc_n": int(acc.sum()), "acc_err": int((acc & err).sum()),
            "n_err": int(err.sum()), "n_attack": int(att.sum()),
            "acc_attack": int((acc & att).sum()),
            "acc_missed_attack": int((acc & att & (pred == 0)).sum()),
            "acc_benign": int((acc & ~att).sum()),
            "acc_false_alarm": int((acc & ~att & (pred != 0)).sum())}

def max_abs_diff(a, b):
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    both_nan = np.isnan(a) & np.isnan(b)
    d = np.abs(a - b); d[both_nan] = 0.0
    return float(np.nanmax(d)) if d.size else 0.0

json.dump(manifest, open(f"{OUT}/R0_environment_and_artifact_manifest.json", "w"), indent=2)
print("[R0] Manifest saved:", f"{OUT}/R0_environment_and_artifact_manifest.json")
print({k: manifest[k] for k in ["python", "numpy", "pandas", "sklearn", "scipy", "xgboost"]})

### CELL R1b — source-null ATC/outcome file (completes Round-1 R1)

In [ ]:
# CELL R1b — source-null ATC/outcome file (completes Round-1 R1)
if "pc" not in globals():
    cal = pd.read_parquet(P["cal"]); yc = labels_from_frame(cal)
    pc = predict(cal[FEATURE_ORDER].to_numpy(np.float64)); del cal; gc.collect()
ATC = {}
for t in ["binary", "multiclass"]:
    conf = 1.0 - pc[t][0]; e = float(np.mean(pc[t][1] != yc[t]))
    ATC[t] = {"source_error": e, "conf_threshold": float(np.quantile(conf, e))}
n_cal = len(yc["binary"]); assert n_cal == 274787
perm_null = np.random.default_rng(42).permutation(n_cal)[:54 * WINDOW].reshape(54, WINDOW)
null_rows = []
for w in range(54):
    idx = perm_null[w]; rest = np.ones(n_cal, bool); rest[idx] = False
    rec = {"window_id": w}
    for t in ["binary", "multiclass"]:
        risk, pred, y = pc[t][0], pc[t][1], yc[t]; conf = 1.0 - risk
        thr_lwo = float(np.quantile(conf[rest], float(np.mean(pred[rest] != y[rest]))))
        rec[f"{t}_atc_est_lwo"]  = float(np.mean(conf[idx] < thr_lwo))
        rec[f"{t}_atc_est_full"] = float(np.mean(conf[idx] < ATC[t]["conf_threshold"]))
        rec[f"{t}_err_rate"] = float(np.mean(pred[idx] != y[idx]))
        for st in ["normal", "elevated", "severe"]:
            r = window_record(risk[idx], pred[idx], y[idx], THR[st][t])
            rec[f"{t}_{st}_coverage"] = r["acc_n"] / r["n"]; rec[f"{t}_{st}_acc_err"] = r["acc_err"]
    null_rows.append(rec)
null_df = pd.DataFrame(null_rows)
null_df.to_csv(f"{OUT}/R1_source_null_windows_ATC_and_outcomes.csv", index=False)
json.dump({"ATC": ATC, "posthoc": True}, open(f"{OUT}/R1_parameters.json", "w"), indent=2)
print("[R1b] COMPLETE", null_df.shape)

# Threshold set (same definition as Round-1 R1; rebuilt here so R6/R7 can run after a restart)
COV_GRID = [0.999, 0.995, 0.99, 0.98, 0.975, 0.95, 0.925, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
THRESH_SET = {f"state_{st}": {t: THR[st][t] for t in ["binary", "multiclass"]} for st in ["normal", "elevated", "severe"]}
for c in COV_GRID:
    THRESH_SET[f"cov_{c:.3f}"] = {t: float(np.sort(pc[t][0])[int(np.ceil(c * n_cal)) - 1]) for t in ["binary", "multiclass"]}
_ref = pd.read_csv(f"{OUT}/R1_threshold_set_calibration.csv")
for _, r in _ref.iterrows():
    assert THRESH_SET[r.thr_name][r.task] == r.threshold, ("threshold set drift", r.thr_name, r.task)
print("[R1b] Threshold set identical to Round 1: PASS")

### CELL R6 — Level-2 compound shifts: reproduction vs V2.11F-C + per-window outcomes for every threshold

In [ ]:
# CELL R6 — Level-2 compound shifts: reproduction vs V2.11F-C + per-window outcomes for every threshold
L2DIR = f"{V2}/controlled_shifts/Level2_compounds/reconstructed_features"
SCEN = {
  "S3_corruption_10_PLUS_S4_important_missing_10": ("S3_corruption_10_PLUS_S4_important_missing_10_V2_deterministic_float64_features.npz", "S34", 82, 412181, 410000),
  "S3_corruption_30_PLUS_S4_important_missing_30": ("S3_corruption_30_PLUS_S4_important_missing_30_V2_deterministic_float64_features.npz", "S34", 82, 412181, 410000),
  "S3_corruption_10_PLUS_S5_attack_enriched":      ("S3_corruption_10_PLUS_S5_attack_enriched_V2_deterministic_features.npz", "S5", 28, 142779, 140000),
  "S3_corruption_30_PLUS_S5_attack_enriched":      ("S3_corruption_30_PLUS_S5_attack_enriched_V2_deterministic_features.npz", "S5", 28, 142779, 140000),
  "S4_important_missing_10_PLUS_S5_attack_enriched": ("S4_important_missing_10_PLUS_S5_attack_enriched_exact_features.npz", "S5", 28, 142779, 140000),
  "S4_important_missing_30_PLUS_S5_attack_enriched": ("S4_important_missing_30_PLUS_S5_attack_enriched_exact_features.npz", "S5", 28, 142779, 140000),
}
l2 = pd.read_csv(P["l2_out"])
iid_lab = labels_from_frame(pd.read_parquet(P["iid"], columns=["binary_label", "attack_family"]))
rows, worst = [], 0.0
for scen, (fname, route, nw, npop, nused) in SCEN.items():
    z = np.load(f"{L2DIR}/{fname}", allow_pickle=False)
    X = np.asarray(z["X"], np.float64); src = np.asarray(z["source_row_index"], np.int64); z.close()
    assert X.shape == (npop, 68)
    if route == "S34":
        assert np.array_equal(src, np.arange(npop)); y = iid_lab
        order = np.random.default_rng(42).permutation(npop)[:nused]
    else:
        assert len(np.unique(src)) == npop; y = {t: iid_lab[t][src] for t in iid_lab}
        order = np.arange(nused)
    pr = predict(X); del X
    fz = l2[l2.scenario == scen].sort_values("window_id").reset_index(drop=True); assert len(fz) == nw
    for w in range(nw):
        idx = order[w * WINDOW:(w + 1) * WINDOW]; state = str(fz.loc[w, "state"]).lower()
        base = {"dataset": "Level2", "condition": scen, "window_id": w, "SSS": float(fz.loc[w, "SSS_primary"]), "state": state}
        for t in ["binary", "multiclass"]:
            risk, pred, yy = pr[t][0][idx], pr[t][1][idx], y[t][idx]
            atc = float(np.mean((1.0 - risk) < ATC[t]["conf_threshold"]))
            for name, thrs in THRESH_SET.items():
                rows.append({**base, "task": t, "thr_name": name, "threshold": thrs[t], "atc_est": atc,
                             **window_record(risk, pred, yy, thrs[t])})
            for pol, thr in [("fixed", THR["normal"][t]), ("adaptive", THR[state][t])]:
                r = window_record(risk, pred, yy, thr)
                cov = r["acc_n"] / r["n"]; rk = r["acc_err"] / r["acc_n"] if r["acc_n"] else np.nan
                worst = max(worst, max_abs_diff([cov, rk], [fz.loc[w, f"{t}_{pol}_coverage"], fz.loc[w, f"{t}_{pol}_risk"]]))
    del pr; gc.collect(); print(f"  {scen}: done")
print("[R6] REPRODUCTION vs frozen V2.11F-C:", "PASS" if worst < 1e-12 else f"FAIL (max diff {worst:.3e})")
pd.DataFrame(rows).to_csv(f"{OUT}/R6_Level2_window_threshold_outcomes.csv.gz", index=False)
print("[R6] COMPLETE")

### CELL R7 — CSE-CIC-IDS2018: security metrics (missed attacks, false alarms, attack deferral) + shard composition

In [ ]:
# CELL R7 — CSE-CIC-IDS2018: security metrics (missed attacks, false alarms, attack deferral) + shard composition
import pyarrow.parquet as pq
shards = sorted(os.path.join(P["s1_pred_dir"], f) for f in os.listdir(P["s1_pred_dir"]) if f.endswith(".parquet"))
pf0 = pq.ParquetFile(shards[0]); schema = pf0.schema_arrow
print("[R7] Shard files:", [os.path.basename(s) for s in shards])
print("[R7] Schema:"); [print("     ", f.name, f.type) for f in schema]
print("[R7] First row:", pf0.read_row_group(0).slice(0, 1).to_pylist())
names = schema.names
BIN_CANDS = ["binary_label", "y_true_binary", "binary_true", "true_binary", "y_binary", "binary_y_true", "y_true_b"]
FAM_CANDS = ["attack_family", "true_family", "family", "multiclass_label", "y_true_multiclass", "multiclass_true", "raw_label", "Label", "label"]
bcol = next((c for c in BIN_CANDS if c in names), None)
fcol = next((c for c in FAM_CANDS if c in names), None)
print("[R7] binary label column:", bcol, "| family/raw label column:", fcol)
if bcol is None and fcol is None:
    raise RuntimeError("No label column found in S1 shards — paste the schema above and STOP; I will adapt the cell.")

need = ["binary_error", "binary_risk"] + [c for c in [bcol, fcol] if c]
data = {c: [] for c in need}; shard_id = []
for i, sp in enumerate(shards):
    tb = pq.read_table(sp, columns=need)
    for c in need:
        data[c].append(tb.column(c).to_pandas().to_numpy())
    shard_id.append(np.full(tb.num_rows, i + 1, np.int16))
data = {c: np.concatenate(v) for c, v in data.items()}; shard_id = np.concatenate(shard_id)
N = len(shard_id); assert N == 16_232_943

def to_binary(v):
    s = pd.Series(v)
    if np.issubdtype(s.dtype, np.number):
        return s.to_numpy(np.int64)
    s = s.astype(str).str.strip().str.lower()
    return np.where(s.isin(["benign", "0", "normal"]), 0, 1).astype(np.int64)

y_b = to_binary(data[bcol]) if bcol else to_binary(data[fcol])
err_b = data["binary_error"].astype(bool)
pred_b = np.where(err_b, 1 - y_b, y_b)           # binary prediction recovered from label and error flag
risk_b = data["binary_risk"].astype(np.float64)
print("[R7] Attack prevalence:", round(float(y_b.mean()), 6), "| binary error rate:", round(float(err_b.mean()), 6))

# Shard composition (which capture day / attack families each shard holds)
comp_col = fcol if fcol else bcol
comp = pd.crosstab(shard_id, pd.Series(data[comp_col]).astype(str))
comp.index.name = "shard"
comp.to_csv(f"{OUT}/R7_S1_shard_label_composition.csv")
print(comp.to_string())

s1_mon = pd.read_csv(P["s1_monitor"]).sort_values("window_id").reset_index(drop=True)
n_win = N // WINDOW
rows = []
fam = pd.Series(data[fcol]).astype(str).to_numpy() if fcol else None
for w in range(n_win):
    a, b = w * WINDOW, (w + 1) * WINDOW
    base = {"window_id": w, "first_shard": int(shard_id[a]), "state": str(s1_mon.loc[w, "primary_shift_evidence_state"]).lower(),
            "dominant_label": (pd.Series(fam[a:b]).value_counts().index[0] if fam is not None else "")}
    for name, thrs in THRESH_SET.items():
        rows.append({**base, "task": "binary", "thr_name": name,
                     **window_record(risk_b[a:b], pred_b[a:b], y_b[a:b], thrs["binary"])})
out = pd.DataFrame(rows)
# consistency gate with Round-1 R4 (accepted errors must match exactly)
r4 = pd.read_csv(f"{OUT}/R4_S1_window_threshold_outcomes.csv.gz")
chk = out.merge(r4[r4.task == "binary"][["window_id", "thr_name", "acc_n", "acc_err"]], on=["window_id", "thr_name"], suffixes=("", "_r4"))
ok = (chk.acc_n == chk.acc_n_r4).all() and (chk.acc_err == chk.acc_err_r4).all()
print("[R7] Consistency with Round-1 R4:", "PASS" if ok else "FAIL")
out.to_csv(f"{OUT}/R7_S1_binary_security_window_outcomes.csv.gz", index=False)
tot = out.groupby("thr_name")[["n", "acc_n", "n_attack", "acc_attack", "acc_missed_attack", "acc_benign", "acc_false_alarm"]].sum()
tot["coverage"] = tot.acc_n / tot.n; tot["attack_deferred"] = 1 - tot.acc_attack / tot.n_attack
tot["miss_rate_accepted_attacks"] = tot.acc_missed_attack / tot.acc_attack; tot["false_alarm_rate_accepted_benign"] = tot.acc_false_alarm / tot.acc_benign
print(tot.loc[["state_normal", "state_elevated", "state_severe"], ["coverage", "attack_deferred", "miss_rate_accepted_attacks", "false_alarm_rate_accepted_benign"]].round(4).to_string())
del data; gc.collect()
print("[R7] COMPLETE")

### CELL R8 — package Round-2 outputs

In [ ]:
# CELL R8 — package Round-2 outputs
EXP2 = f"{OUT}/export_R2"; os.makedirs(EXP2, exist_ok=True)
for f in ["R1_source_null_windows_ATC_and_outcomes.csv", "R1_parameters.json", "R6_Level2_window_threshold_outcomes.csv.gz",
          "R7_S1_binary_security_window_outcomes.csv.gz", "R7_S1_shard_label_composition.csv"]:
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy2(f"{OUT}/{f}", f"{EXP2}/{f}")
z = shutil.make_archive(f"{OUT}/TrustShift_revision_R2_export", "zip", EXP2)
print("Upload:", z, f"({os.path.getsize(z)/1e6:.1f} MB)")